<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_review_post_process.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Set up

In [2]:
# @title Dependencies

# Installation
!pip install strsimpy -q

# First-party installations
import os
from google.colab import drive
from IPython.display import display
import re
import hashlib
import json

# Third-party installations
import pandas as pd
import numpy as np
from tqdm import tqdm
from strsimpy.normalized_levenshtein import NormalizedLevenshtein

In [3]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR
DATASET_DIR = os.path.join(WORKING_DIR, "llm_reviews_raw")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews_processed")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet output)
RESULTS_PATH = os.path.join(RAW_DIR, "llm_reviews_segmented.parquet")
# Define the full result path for intermediate JSONL storage
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_segmented.jsonl")

Mounted at /content/drive
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured dataset directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_raw.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_processed.


In [12]:
# @title Setup definitions

INPUT_DATA = "llm_reviews_results.parquet"

TARGET_SECTION = "strenghts_and_weaknesses"

REVIEW_COL_ORIGINAL = 'llm_review'
REVIEW_COL_EXTRACTED = 'extracted_weaknesses'
REVIEW_COL_SEGMENTS = 'segmented_comment'
UNI_ID = "segment_hash"

In [8]:
# @title Data import

# Read dataset
df = pd.read_parquet(os.path.join(DATASET_DIR, INPUT_DATA))

# Check and display
display(df.shape)
display(df.head(3))

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using..."
1,vuD2xEtxZcj,gpt-5-nano-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""The paper investigates N..."
2,vuD2xEtxZcj,gpt-4o-2024-11-20,0.0,None,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{\n ""summary_of_paper"": ""The paper introduces..."


(9, 9)

## Define

In [13]:
# @title Extraction logic

def extract_weaknesses(json_str):
    """
    Parses the json_str and extracts the TARGET_SECTION section.
    Returns a list of strings, or an empty list if parsing fails or the section is missing.
    """
    try:
        # Load columnn data into JSON
        review_data = json.loads(json_str)

        # Isolate the TARGET_SECTION
        target_section = review_data.get(TARGET_SECTION, [])

        # Return the whole section
        return target_section

    except Exception as e:
        # Print a hardcoded error message for debugging
        print(f"Warning: Failed to isolate weaknesses. Error: {e}. Review string: {json_str[:100]}...")
        return []

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review,extracted_weaknesses
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",[Strength: Identifies a clear mismatch between...
1,vuD2xEtxZcj,gpt-5-nano-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""The paper investigates N...",[Strong theoretical motivation for using MVUE ...
2,vuD2xEtxZcj,gpt-4o-2024-11-20,0.0,None,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{\n ""summary_of_paper"": ""The paper introduces...",[Strength: The paper addresses a critical gap ...
3,vuD2xEtxZcj,gpt-4o-mini-2024-07-18,0.0,None,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{\n ""summary_of_paper"": ""This paper presents ...",[The paper introduces a significant advancemen...
4,vuD2xEtxZcj,o4-mini-2025-04-16,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper extends fine-...","[Strengths:, - Identifies a critical failure o..."


(9, 10)

## Run

In [ ]:
# @title Extraction

# Apply the extraction function
df[REVIEW_COL_EXTRACTED] = df[REVIEW_COL_ORIGINAL].apply(extract_weaknesses)

# Check and display extracted section(s)
display(df.shape)
display(df.head())

## Transformation

In [14]:
# @title Long-format

# Create a copy of the extraction column
df[REVIEW_COL_SEGMENTS] = df[REVIEW_COL_EXTRACTED]

# Convert the copied extraction column into long-format
df = df.explode(REVIEW_COL_SEGMENTS)

# Display and check segmented comment(s)
display(df.shape)
display(df.head())

,paper_id,model,temperature,reasoning_effort,chain_of_thought,note_taking,run_signature,llm_notes,llm_review,segmented_comment
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",Strength: Identifies a clear mismatch between ...
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",Strength: Derives analytically grounded MVUE s...
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",Strength: Empirical results across models and ...
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...","Strength: Provides implementation details, com..."
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,,Turned off,45d644307435,NOTES_SKIPPED_FULL_TEXT_USED,"{""summary_of_paper"": ""This paper studies using...",Weakness: The optimal 2:4 MVUE is reported as ...


(60, 10)

In [ ]:
# @title Unique identifiers

# Get all columns as basis for unique identifier (hash)
col_for_hash = df.columns.tolist()

# Create a unique identifier (hash)
df[UNI_ID] = df[col_for_hash].astype(str).agg(''.join, axis=1).apply(lambda x: hashlib.sha1(x.encode('utf-8')).hexdigest()[:12])

# Display and check unique identifier(s)
display(df.shape)
display(df.head())

In [ ]:
# @title Results

# Save the long-format DataFrame to JSONL
df.to_json(RESULTS_JSON_PATH, orient='records', lines=True)
print(f"DataFrame saved to JSONL at: {RESULTS_JSON_PATH}")

# Save the long-format DataFrame to Parquet
df.to_parquet(RESULTS_PATH, index=False)
print(f"DataFrame saved to Parquet at: {RESULTS_PATH}")